# Week 2 — Apparel & Fashion Retail EDA
This notebook reproduces the cleaning, profiling, descriptive statistics, grouped summaries, outlier checks, and visualizations in the report. The source metadata describes the 3,400-row data as simulated.

## 1. Imports and loading
Run the notebook from the repository root. The source CSV is fetched directly from the linked public dataset repository.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOURCE_URL = 'https://raw.githubusercontent.com/Ishita95-AIML/-Fashion-Retail-Sales-Dataset/main/Fashion_Retail_Sales.csv'
raw = pd.read_csv(SOURCE_URL, dtype={'Customer Reference ID': 'string'})
df = raw.copy()
df['Purchase Amount (USD)'] = pd.to_numeric(df['Purchase Amount (USD)'], errors='coerce')
df['Review Rating'] = pd.to_numeric(df['Review Rating'], errors='coerce')
df['Date Purchase'] = pd.to_datetime(df['Date Purchase'], format='%d-%m-%Y', errors='coerce')
print('Shape:', df.shape)
df.head()

## 2. Dataset structure and missing values
Missing values are kept as missing. In particular, a missing amount is not assumed to be a zero purchase.

In [ ]:
display(df.info())
missing = df.isna().sum().to_frame('missing_rows')
missing['missing_rate_pct'] = (missing['missing_rows'] / len(df) * 100).round(2)
display(missing)
print('Date range:', df['Date Purchase'].min(), 'to', df['Date Purchase'].max())
print('Rating range:', df['Review Rating'].min(), 'to', df['Review Rating'].max())

## 3. Descriptive statistics and outlier assessment
Use quartiles and the 1.5×IQR upper fence. Flag potential outliers for investigation instead of deleting them automatically.

In [ ]:
amount = df['Purchase Amount (USD)'].dropna()
q1, median, q3 = amount.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
amount_summary = amount.describe(percentiles=[0.25, 0.50, 0.75])
display(amount_summary.to_frame('purchase_amount_usd'))
print('IQR upper fence:', upper_fence)
print('IQR outlier count:', int((amount > upper_fence).sum()))
print('Amount > USD 1,000:', int((amount > 1000).sum()))
print('Rating summary:')
display(df['Review Rating'].describe().to_frame('rating'))

## 4. Product and payment mix
Counts refer to rows in this CSV, not units, because there is no quantity field.

In [ ]:
top_items = df['Item Purchased'].value_counts().head(10)
display(top_items.to_frame('transaction_rows'))
payment_counts = df['Payment Method'].value_counts()
display(payment_counts.to_frame('transaction_rows'))
payment_summary = df.groupby('Payment Method').agg(
    transaction_rows=('Item Purchased', 'size'),
    amount_present=('Purchase Amount (USD)', 'count'),
    mean_observed_amount=('Purchase Amount (USD)', 'mean'),
    mean_review_rating=('Review Rating', 'mean'),
)
display(payment_summary)

## 5. Monthly summaries
Monthly sums exclude missing amounts, so they are observed totals rather than audited revenue. Keep row counts and amount-present counts beside the sum.

In [ ]:
df['Month'] = df['Date Purchase'].dt.to_period('M')
monthly = df.groupby('Month').agg(
    transaction_rows=('Item Purchased', 'size'),
    amount_present=('Purchase Amount (USD)', 'count'),
    observed_amount_sum=('Purchase Amount (USD)', 'sum'),
)
display(monthly)
ax = monthly['observed_amount_sum'].plot(marker='o', figsize=(9, 4), title='Observed purchase-amount total by month')
ax.set_ylabel('USD (non-missing amounts only)')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 6. Visualizations
The box plot displays the central amount range without the highly separated extreme values; the count of flagged values is reported explicitly. The bar chart compares item-name frequency, and the donut chart shows the two payment categories.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.boxplot(amount, vert=False, showfliers=False)
ax.set_title('Purchase amount distribution — central range')
ax.set_xlabel('USD')
ax.grid(axis='x', alpha=0.25)
ax.text(0.02, -0.25, f'{int((amount > upper_fence).sum())} values exceed the 1.5×IQR upper fence', transform=ax.transAxes)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
top_items.sort_values().plot(kind='barh', ax=ax, title='Top item names by transaction-row count')
ax.set_xlabel('Rows'); ax.grid(axis='x', alpha=0.25)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 4.5))
payment_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%', ylabel='', title='Payment method mix', wedgeprops={'width':0.42})
plt.tight_layout(); plt.show()

## 7. Correlation and interpretation
Only rows with both numeric fields present are included. The result describes a linear association, not a causal effect.

In [ ]:
complete = df[['Purchase Amount (USD)', 'Review Rating']].dropna()
print('Complete cases:', len(complete))
print('Pearson correlation:', complete['Purchase Amount (USD)'].corr(complete['Review Rating']))

## 8. Business takeaways and limitations
- Verify the extreme purchase amounts with order/receipt records before revenue reporting.
- Investigate the reasons for missing amounts and ratings; do not replace blanks with zero by default.
- Use item-row counts only as a preliminary signal until units, inventory, margins, returns, and category taxonomy are available.
- Treat the last month as partial and obtain a longer time series before seasonal claims.
- The metadata describes the dataset as simulated; the findings are an educational baseline, not verified performance of a real retailer.